# PCAP Statistics

Extração completa do diretório **01-12**, independente dos CSVs do CICFlowMeter.
Cada linha representa uma janela global de todo o tráfego IPv4 observado, sem agrupamento por IP.
Não há rotulagem, seleção de ataques, balanceamento, imputação ou criação de cenários nesta etapa.
A referência desta implementação é o projeto `Atual-2.zip`.


## 1. Dependencies and Configuration

Execute no computador com acesso ao disco E:. Se necessário, descomente a instalação e execute-a uma vez.
`maxPackets=None` processa todos os pacotes de todos os arquivos; um inteiro limita globalmente os pacotes processados, incluindo não IPv4.
A intercalação mantém um pacote de antecipação por arquivo, fora da contagem de observações processadas.


In [5]:
# %pip install -r src/PcapStatistics/requirements.txt

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "PcapStatistics").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Execute este notebook dentro da raiz do projeto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.PcapStatistics import PcapStatisticsDataset, FEATURE_NAMES, discover_pcaps

PCAP_DIR = Path(r"E:\CICDDoS2019\01-12\PCAP")
OUTPUT_DIR = PROJECT_ROOT / "output" / "PcapStatistics" / "01-12"
WINDOW_SECONDS = 1.0  # Pode mudar para 5.0, 10.0 etc.
MAX_PACKETS = None    # None: leitura completa. Exemplo de teste: 100_000.
FLOW_IDLE_TIMEOUT_SECONDS = 600.0
SUBFLOW_GAP_SECONDS = 1.0
PACKET_LENGTH_MODE = "transport_payload"  # Alternativa explícita: "ip".

extractor = PcapStatisticsDataset(
    windowSeconds=WINDOW_SECONDS,
    maxPackets=MAX_PACKETS,
    flowIdleTimeoutSeconds=FLOW_IDLE_TIMEOUT_SECONDS,
    subflowGapSeconds=SUBFLOW_GAP_SECONDS,
    packetLengthMode=PACKET_LENGTH_MODE,
    keepEmptyWindows=True,
    progressEvery=1_000_000,
)


## 2. Input Files

A descoberta identifica PCAP/PCAPNG pelo cabeçalho binário, incluindo arquivos sem extensão como `SAT-01-12-2018_0`.
Todos os arquivos encontrados são intercalados cronologicamente; a ordem dos nomes não define a ordem dos pacotes.
O estado das conexões é preservado entre arquivos. Sobreposições são intercaladas e nenhum pacote é deduplicado.
Cada arquivo deve estar internamente ordenado: regressões de timestamp interrompem a execução.


In [6]:
pcapFiles = discover_pcaps(PCAP_DIR)
print(f"Arquivos PCAP/PCAPNG: {len(pcapFiles)} | features: {len(FEATURE_NAMES)} | janela: {WINDOW_SECONDS:g}s")


Arquivos PCAP/PCAPNG: 819 | features: 44 | janela: 1s


## 3. Global Statistical Dataset

Uma leitura atualiza contadores, médias e dispersões por pacote. O CSV é gravado progressivamente, sem manter o dataset inteiro em memória.
As janelas seguem o epoch UTC: `[início, fim)`. O timestamp do dataset corresponde ao término da janela; são preservadas as datas reais dos PCAPs.
O intervalo exportado vai da primeira à última observação processada. Janelas sem pacotes entre essas bordas são mantidas; não se inventam horas antes/depois da captura.
Janelas vazias não comprovam disponibilidade do sensor: podem representar silêncio ou lacunas de captura.

As 33 features solicitadas são acompanhadas por **6 contagens de valores únicos e 5 entropias** (44 features).
O CSV contém exclusivamente `Timestamp` e as 44 features. A duração da janela está na configuração e no relatório de fórmulas; contadores auxiliares são internos.

**Definições temporais:**

- Forward/backward são determinados pela primeira orientação observada de cada sessão, sem inferir cliente/servidor por porta ou rótulo.
- `Flow_Duration` é a média da duração dos trechos das sessões dentro da janela; IATs usam pares da mesma sessão e janela.
- Contagens, bytes, cabeçalhos e taxas são da janela atual; as taxas usam sua duração nominal.
- As janelas TCP iniciais preservam o primeiro valor observado de cada direção, sem conhecer pacotes futuros.
- `Subflow_*` soma, por sessão na janela, os respectivos totais divididos pela quantidade de subfluxos locais. Com 1s e limiar de gap de 1s, pode coincidir com os totais.
- O modo padrão de comprimento usa payload de transporte; `ip` usa comprimento IPv4 capturado. A escolha consta no relatório.
- Fragmentos permanecem nas contagens IP e entropias, mas não recebem direção TCP/UDP sem remontagem. Comprimentos desconhecidos ficam ausentes.
- Ausências estatísticas são campos vazios; contadores e somas vazios são zero. Não há imputação.

São adaptações documentadas para observação temporal, sem promessa de equivalência numérica com registros finais do CICFlowMeter.


In [7]:
result = extractor.runFiles(
    pcapFiles=pcapFiles,
    outputPath=OUTPUT_DIR / f"global_windows_{WINDOW_SECONDS:g}s.csv",
    featureReportPath=OUTPUT_DIR / "feature_calculations.csv",
)
print(f"Concluído: {result.packetsProcessed:,} pacotes processados | {result.ipv4Packets:,} IPv4 | {result.windowsWritten:,} janelas")


1,000,000 pacotes processados; 4,739 janelas gravadas.
2,000,000 pacotes processados; 4,744 janelas gravadas.
3,000,000 pacotes processados; 4,749 janelas gravadas.
4,000,000 pacotes processados; 4,754 janelas gravadas.
5,000,000 pacotes processados; 4,758 janelas gravadas.
6,000,000 pacotes processados; 4,763 janelas gravadas.
7,000,000 pacotes processados; 4,768 janelas gravadas.
8,000,000 pacotes processados; 4,772 janelas gravadas.
9,000,000 pacotes processados; 4,778 janelas gravadas.
10,000,000 pacotes processados; 4,784 janelas gravadas.
11,000,000 pacotes processados; 4,791 janelas gravadas.
12,000,000 pacotes processados; 4,798 janelas gravadas.
13,000,000 pacotes processados; 4,806 janelas gravadas.
14,000,000 pacotes processados; 4,813 janelas gravadas.
15,000,000 pacotes processados; 4,822 janelas gravadas.
16,000,000 pacotes processados; 4,828 janelas gravadas.
17,000,000 pacotes processados; 4,835 janelas gravadas.
18,000,000 pacotes processados; 4,842 janelas gravadas.
1

KeyboardInterrupt: 

## 4. Dataset Inspection and Feature Calculations

A inspeção lê somente as primeiras linhas do CSV final. O único relatório gerado é `feature_calculations.csv`, com nome, fórmula, unidade e convenções de cada coluna.
A primeira e a última linha podem representar janelas incompletas. Não são exportadas colunas de borda ou diagnóstico. O resumo da execução permanece em memória, sem relatório adicional.
Para a extração completa mantenha `MAX_PACKETS=None`. Reinicie o kernel após atualizar os módulos e execute o notebook em ordem.


In [ ]:
display(pd.read_csv(result.datasetPath, nrows=5))
featureCalculations = pd.read_csv(result.featureReportPath)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(featureCalculations.loc[featureCalculations["Kind"].eq("feature"), ["Feature", "Formula", "Unit", "Notes"]])
print("Dataset:", result.datasetPath)
print("Cálculos:", result.featureReportPath)


,Timestamp,ACK_Flag_Count,Average_Packet_Size,Avg_Fwd_Segment_Size,Bwd_Header_Length,Bwd_Packets_s,CWE_Flag_Count,Flow_Bytes_s,Flow_Duration,Flow_IAT_Max,...,UniqueDestinationIPs,UniqueSourcePorts,UniqueDestinationPorts,UniqueProtocols,UniqueFlows,SourceIPEntropy,DestinationIPEntropy,SourcePortEntropy,DestinationPortEntropy,ProtocolEntropy
0,2018-12-01T13:17:11.000000000Z,0,NaN,NaN,0,0.0,0,0.0,NaN,NaN,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN
1,2018-12-01T13:17:12.000000000Z,22,239.153846,120.50,352,10.0,0,6218.0,108064.500000,192825.0,...,2,3,3,1,2,0.961237,0.961237,1.460485,1.238901,-0.000000
2,2018-12-01T13:17:13.000000000Z,2,0.000000,0.00,0,0.0,0,0.0,3.000000,3.0,...,1,1,1,1,1,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
3,2018-12-01T13:17:14.000000000Z,6,24.200000,30.25,64,2.0,0,242.0,11013.333333,33023.0,...,4,3,3,2,3,1.921928,1.921928,1.584963,1.584963,0.970951
4,2018-12-01T13:17:15.000000000Z,12,0.500000,1.00,192,6.0,0,6.0,27272.333333,33130.0,...,4,5,5,1,3,1.792481,1.792481,2.251629,2.251629,-0.000000


,Feature,Formula,Unit,Notes
0,ACK_Flag_Count,sum(1[TCP.flags & 0x10]),packets,"Pacotes com ACK, em ambas as direções."
1,Average_Packet_Size,sum(L) / quantidade de L disponíveis,bytes,Igual a Packet_Length_Mean; não reproduz duplicações legadas de extratores.
2,Avg_Fwd_Segment_Size,sum(L_forward) / quantidade de L_forward disponíveis,bytes,Alias de Fwd_Packet_Length_Mean.
3,Bwd_Header_Length,sum(comprimento cabeçalho transporte de pacotes backward),bytes,Mesma convenção de Fwd_Header_Length.
4,Bwd_Packets_s,Total_Backward_Packets / WindowSeconds,packets/s,NaN
5,CWE_Flag_Count,sum(1[TCP.flags & 0x80]),packets,"Nome legado CWE corresponde a CWR, não ECE."
6,Flow_Bytes_s,sum(L disponíveis na janela) / WindowSeconds,bytes/s,Denominador fixo da janela; não duração final dos fluxos.
7,Flow_Duration,mean_f((último_timestamp_f - primeiro_timestamp_f) / 1000),microseconds,f = trecho da sessão dentro desta janela; um pacote implica duração zero. Não é idade nem duração final da conexão.
8,Flow_IAT_Max,max de (t_i - t_anterior_mesma_sessão) / 1000,microseconds,Somente pares da mesma sessão e mesma janela; exclui intervalos entre janelas e conexões distintas.
9,Flow_IAT_Mean,sum(IATs intrassessão na janela) / número desses intervalos,microseconds,Média agrupada por intervalo; sem pares entre conexões ou janelas.


Dataset: C:\GitHub\anomaly-detection-data-stream\output\PcapStatistics\01-12\global_windows_1s.csv
Cálculos: C:\GitHub\anomaly-detection-data-stream\output\PcapStatistics\01-12\feature_calculations.csv
